https://www.kaggle.com/datasets/youssefsalahzakria/fruit-and-vegetables-classification

Although I initally wanted to repurpose the classification data to use in this classification model, many of the includes samples would lead to confusion within the task of classification due to the inclusion of other objects and fruits being shown in different forms than their intended use. e.g popscicles or on a platter with other items. THis datase5t, very clearly lays out what the classifier is looking for in several different contexts.

In [4]:
import os
import shutil
import random
from PIL import Image
import numpy as np
import tensorflow as tf
from tqdm import tqdm

In [6]:
BASE_DIR = r"C:\Users\DM77\Documents\FruitVisionRT\data\classification\Fruit and Vegetables"
CLASSES = [
    'apple', 'banana', 'orange', 'strawberry', 'grapes', 'watermelon', 'lemon',
    'kiwi', 'mango', 'peach', 'pineapple', 'pomegranate', 'tomato', 'avocado',
    'cucumber', 'pear', 'cherry'
]
IMG_SIZE = (128, 128)
TARGET_COUNT = 600

In [7]:
def augment_tf(img_array):
    img = tf.convert_to_tensor(img_array, dtype=tf.uint8)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    return img.numpy()

In [8]:
for split in ['train']:
    for cls in CLASSES:
        cls_path = os.path.join(BASE_DIR, split, cls.capitalize())
        if not os.path.exists(cls_path):
            continue

        imgs = [f for f in os.listdir(cls_path) if f.endswith(".jpg")]
        needed = TARGET_COUNT - len(imgs)

        print(f"{cls.capitalize()}: {len(imgs)} → {TARGET_COUNT} (augmenting {needed})")

        for i in range(needed):
            base_img = Image.open(os.path.join(cls_path, random.choice(imgs))).convert("RGB").resize(IMG_SIZE)
            aug_img = Image.fromarray(augment_tf(np.array(base_img)))
            aug_img.save(os.path.join(cls_path, f"aug_{i}.jpg"))

Apple: 1151 → 600 (augmenting -551)
Banana: 1740 → 600 (augmenting -1140)
Orange: 2592 → 600 (augmenting -1992)
Strawberry: 1305 → 600 (augmenting -705)
Grapes: 1710 → 600 (augmenting -1110)
Watermelon: 1238 → 600 (augmenting -638)
Lemon: 1266 → 600 (augmenting -666)
Kiwi: 942 → 600 (augmenting -342)
Mango: 1140 → 600 (augmenting -540)
Pineapple: 1653 → 600 (augmenting -1053)
Pomegranate: 2337 → 600 (augmenting -1737)
Tomato: 715 → 600 (augmenting -115)
Avocado: 581 → 600 (augmenting 19)
Cucumber: 1424 → 600 (augmenting -824)
Pear: 1088 → 600 (augmenting -488)


In [ ]:
# After seeing most classes are overrepresented, I decided to actually apoply further augmentations to overrepresented classes as underrepresented ones already have appropriate augmentation and the overrepresented ones were taken at face value in a way

In [9]:
from PIL import ImageEnhance, ImageFilter

AUG_SAMPLE_COUNT = 100  # Number of extra augmentations for well-represented classes


def controlled_augment(img):

    img = img.rotate(random.uniform(-25, 25))  # random rotation
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(random.uniform(0.75, 1.25))  # contrast
    enhancer = ImageEnhance.Color(img)
    img = enhancer.enhance(random.uniform(0.8, 1.2))  # color saturation
    img = img.transform(img.size, Image.AFFINE,
                        (1, 0, random.uniform(-5, 5), 0, 1, random.uniform(-5, 5)))  # translation
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0, 1.2)))  # occasional blur
    return img


for cls in CLASSES:
    cls_path = os.path.join(BASE_DIR, 'train', cls.capitalize())
    if not os.path.exists(cls_path):
        continue

    imgs = [f for f in os.listdir(cls_path) if f.endswith(".jpg")]

    if len(imgs) > TARGET_COUNT:
        print(f" Augmenting {cls.capitalize()} with {AUG_SAMPLE_COUNT} extra images...")
        for i in range(AUG_SAMPLE_COUNT):
            base_img = Image.open(os.path.join(cls_path, random.choice(imgs))).convert("RGB").resize(IMG_SIZE)
            aug_img = controlled_augment(base_img)
            aug_img.save(os.path.join(cls_path, f"aug2_{i}.jpg"))


 Augmenting Apple with 100 extra images...
 Augmenting Banana with 100 extra images...
 Augmenting Orange with 100 extra images...
 Augmenting Strawberry with 100 extra images...
 Augmenting Grapes with 100 extra images...
 Augmenting Watermelon with 100 extra images...
 Augmenting Lemon with 100 extra images...
 Augmenting Kiwi with 100 extra images...
 Augmenting Mango with 100 extra images...
 Augmenting Pineapple with 100 extra images...
 Augmenting Pomegranate with 100 extra images...
 Augmenting Tomato with 100 extra images...
 Augmenting Cucumber with 100 extra images...
 Augmenting Pear with 100 extra images...


In [16]:
import os

def get_class_counts(base_dir, splits=['train', 'validation', 'test']):
    class_counts = {split: {} for split in splits}
    for split in splits:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            continue
        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            if not os.path.isdir(cls_path): continue
            count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            class_counts[split][cls] = count
    return class_counts

# Run this block
BASE_DIR = r"C:\Users\DM77\Documents\FruitVisionRT\data\classification\Fruit and Vegetables"
counts = get_class_counts(BASE_DIR)

# Print total counts and find min per split
print(" Final Class Counts:")
min_counts = {}
for split, class_dict in counts.items():
    print(f"\n{split.upper()}:")
    min_count = min(class_dict.values())
    min_counts[split] = min_count
    for cls, count in sorted(class_dict.items()):
        print(f"  {cls.capitalize():<12} — {count} images")
    print(f"🔸 Lowest class count: {min_count}")


 Final Class Counts:

TRAIN:
  Apple        — 1421 images
  Avocado      — 1060 images
  Banana       — 1966 images
  Cucumber     — 1542 images
  Grapes       — 1840 images
  Kiwi         — 1252 images
  Lemon        — 1384 images
  Mango        — 1480 images
  Orange       — 2797 images
  Pear         — 1214 images
  Pineapple    — 1768 images
  Pomegranate  — 2442 images
  Strawberry   — 1579 images
  Tomato       — 823 images
  Watermelon   — 1524 images
🔸 Lowest class count: 823

VALIDATION:
  Apple        — 88 images
  Avocado      — 280 images
  Banana       — 596 images
  Cucumber     — 234 images
  Grapes       — 445 images
  Kiwi         — 557 images
  Lemon        — 437 images
  Mango        — 435 images
  Orange       — 475 images
  Pear         — 157 images
  Pineapple    — 456 images
  Pomegranate  — 383 images
  Strawberry   — 455 images
  Tomato       — 310 images
  Watermelon   — 236 images
🔸 Lowest class count: 88

TEST:
  Apple        — 435 images
  Avocado      — 46

In [ ]:
# We will balance our train data to 820 in ordewr to best represent all classes in another script called balance_train_classes.py

In [17]:
#normal;izing pixel values
from PIL import Image
import numpy as np


def normalize_and_overwrite_images(base_dir, splits=['train', 'validation', 'test']):
    print(" Normalizing images to [0.0, 1.0] and saving back as uint8 (if needed)...\n")
    for split in splits:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            print(f"0 Split folder not found: {split_path}")
            continue

        print(f" {split.upper()}:")
        for cls_name in sorted(os.listdir(split_path)):
            cls_path = os.path.join(split_path, cls_name)
            if not os.path.isdir(cls_path):
                continue

            img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            print(f"  1 {cls_name}: {len(img_files)} images")

            for img_file in tqdm(img_files, desc=f"    Normalizing {cls_name}", leave=False):
                img_path = os.path.join(cls_path, img_file)
                try:
                    img = Image.open(img_path).convert("RGB")
                    img_array = np.array(img).astype(np.float32) / 255.0  # Normalize to [0.0, 1.0]


                    img_uint8 = Image.fromarray((img_array * 255).astype(np.uint8))
                    img_uint8.save(img_path)
                except Exception as e:
                    print(f" Error with {img_path}: {e}")

    print("\n Normalization complete for all available splits.")



normalize_and_overwrite_images(BASE_DIR)


 Normalizing images to [0.0, 1.0] and saving back as uint8 (if needed)...

 TRAIN:
  1 Apple: 820 images


    Normalizing Apple:  37%|███▋      | 300/820 [00:02<00:12, 41.03it/s] C:\Users\DM77\Documents\FruitVisionRT\.venv\lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  1 Avocado: 820 images


  1 Banana: 820 images


  1 Cucumber: 820 images


...

  1 Pomegranate: 787 images


  1 Strawberry: 419 images


  1 Tomato: 156 images


  1 Watermelon: 527 images



 Normalization complete for all available splits.
